**Find and Terminate EC2 Instances with SSH Open to the World**

**What this script does:**
1. Finds all security groups that allow SSH (port 22) from anywhere (0.0.0.0/0)
2. Finds all running or stopped EC2 instances using those security groups
3. Terminates those EC2 instances

**Why this matters:** Leaving SSH open to the world is a major security risk. This is the type of automation you'd build with AWS Lambda to keep your environment secure.

**Step 1: Import Modules and Connect to AWS**

In [37]:
import boto3
from datetime import datetime


**Step 2: Create variable to call today's date**

In [38]:
today = datetime.today()

**See what the response looks like**

In [39]:
today

datetime.datetime(2026, 7, 5, 17, 26, 12, 533645)

**Format the date to match the project instruction YYYYMMDD and hold value in anothr variable**

In [40]:
todays_date = today.strftime("%Y%m%d")

**See what the output look like**

In [41]:
todays_date

'20260705'

**Step 3: call boto3 client in aws**

In [42]:

s3_client = boto3.client('s3')

**Set our bucket name to bucket in aws**

In [43]:
bucket_name = "nubby-organise-s3-objects"


In [44]:
import boto3
sts = boto3.client('sts')
print(sts.get_caller_identity())

{'UserId': 'AIDAQFGU3WKDXT6RCOEX4', 'Account': '011183829639', 'Arn': 'arn:aws:iam::011183829639:user/programmatic-user', 'ResponseMetadata': {'RequestId': 'ca45986f-9022-4c8a-8637-7ad4807e57c3', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'ca45986f-9022-4c8a-8637-7ad4807e57c3', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6UzoxNzgzMjY4NzcyODU0OlI6MXhYSU0wUDg=', 'content-type': 'text/xml', 'content-length': '414', 'date': 'Sun, 05 Jul 2026 16:26:12 GMT'}, 'RetryAttempts': 0}}


**Step 4: list all objects in bucket**

In [45]:
list_objects_response = s3_client.list_objects_v2(Bucket=bucket_name)

**See what the response looks like**

In [46]:
list_objects_response

{'ResponseMetadata': {'RequestId': 'ANW71H8T5HV4FGTK',
  'HostId': 'opOiSCD4ciTKL8l154EvMcW7qhO479olQgyF3Ljif8OkP0W2l9sztLE73iSh8ipUmVoFlEusWnG+zsCyghcs4WVRbFiP7KI9',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'opOiSCD4ciTKL8l154EvMcW7qhO479olQgyF3Ljif8OkP0W2l9sztLE73iSh8ipUmVoFlEusWnG+zsCyghcs4WVRbFiP7KI9',
   'x-amz-request-id': 'ANW71H8T5HV4FGTK',
   'date': 'Sun, 05 Jul 2026 16:26:16 GMT',
   'x-amz-bucket-region': 'us-east-1',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'IsTruncated': False,
 'Contents': [{'Key': '20260705/',
   'LastModified': datetime.datetime(2026, 7, 5, 15, 47, 1, tzinfo=tzutc()),
   'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
   'ChecksumAlgorithm': ['CRC32'],
   'ChecksumType': 'FULL_OBJECT',
   'Size': 0,
   'StorageClass': 'STANDARD'},
  {'Key': '20260705/IP Transcript.pdf',
   'LastModified': datetime.datetime(2026, 7, 5, 15, 52, 17, tzinfo=tzutc()),
   'ETag': 

**Get the value of the "contents" key from the response**

In [47]:
get_contents = list_objects_response.get("Contents")

**See what the contents look like**

In [48]:
get_contents

[{'Key': '20260705/',
  'LastModified': datetime.datetime(2026, 7, 5, 15, 47, 1, tzinfo=tzutc()),
  'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 0,
  'StorageClass': 'STANDARD'},
 {'Key': '20260705/IP Transcript.pdf',
  'LastModified': datetime.datetime(2026, 7, 5, 15, 52, 17, tzinfo=tzutc()),
  'ETag': '"3e66cbb00e39ed20e6bd24169098580e"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 40179,
  'StorageClass': 'STANDARD'},
 {'Key': '20260705/Unit 5- Promoting Person-Centered Practice.docx',
  'LastModified': datetime.datetime(2026, 7, 5, 15, 52, 17, tzinfo=tzutc()),
  'ETag': '"9df8d96470f3deed7a76a5db6d875a2f"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 24436,
  'StorageClass': 'STANDARD'},
 {'Key': '20260705/photo copy.jpg',
  'LastModified': datetime.datetime(2026, 7, 5, 15, 47, 4, tzinfo=tzutc()),
  'ETag': '"81d2307225f8ab3a78d8e8b

**Step 5: Filter Out all contentds and put them in a list **

In [49]:
get_all_s3_objects_and_folder_names = []

for item in get_contents:
    s3_object_name = item.get("Key")

    get_all_s3_objects_and_folder_names.append(s3_object_name)

**See the list of all contents**

In [50]:
get_all_s3_objects_and_folder_names

['20260705/',
 '20260705/IP Transcript.pdf',
 '20260705/Unit 5- Promoting Person-Centered Practice.docx',
 '20260705/photo copy.jpg',
 '20260705/photo.jpg']

**Step 6: Create a folder for today's date in s3.**

In [51]:
directory_name = todays_date + "/"

**See the folder created for todays date**

In [52]:
directory_name

'20260705/'

**Step 7: check if folder exist for todays date in the s3 bucket**

In [53]:
if directory_name not in get_all_s3_objects_and_folder_names:
    s3_client.put_object(Bucket=bucket_name, Key=(directory_name))

**Now add objects that was uploade today into the folder baed on 2 conditions: first check if lat modified date match our folder name and then check if pbject doesn t have a / at the end and also delete upload file from bucket since they are in the folder already**

In [54]:
for item in get_contents:
    object_create_date = item.get("LastModified").strftime("%Y%m%d") + "/"
    object_name = item.get("Key")

    if object_create_date == directory_name and "/" not in object_name:
        s3_client.copy_object(Bucket=bucket_name, CopySource=bucket_name+"/"+ object_name, Key=directory_name+object_name)
        s3_client.delete_object(Bucket=bucket_name, Key=object_name)

**Step 8: End of code**